# Pangenome Analysis Notebook

This blueprint shows how to XYZ

Add diagram

Graph genomes and Dr. Benedict Paten's lab at the University of California, Santa Cruz (UCSC)

## Download the data

### FASTQ samples

These samples come from 

In [ ]:
%%sh 



### Graph indices

We will use the HPRC ([Human Pangenome Reference Consortium](https://humanpangenome.org/)) version 1.1 graph. It was constructed using [Minigraph-Cactus](https://github.com/ComparativeGenomicsToolkit/cactus/blob/master/doc/pangenome.md). Specifically, we will need the following files: 
- `.gbz` - The compressed graph 
- `.dist` - The distance index used for computing distances between points in the graph 
- `.min` - The minimizer index used to find minimizer substrings in the graph 

Make data directory (if it doesn't exist)

In [ ]:
%%sh 

mkdir -p data/ref
mkdir -p data/out 

Download the data. This takes XYZ minutes. 

In [ ]:
%%sh 

# Donwload index files
cd data/ref && \
    printf "%s\n" dist min gbz | xargs -I {} \
    wget "https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.{}"

Spider mode enabled. Check if remote file exists.
--2026-02-02 11:10:14--  https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.dist
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 3.5.81.14, 52.92.225.200, 52.92.184.128, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|3.5.81.14|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8694520760 (8.1G) [application/vnd.apple.installer+xml]
Remote file exists.

Spider mode enabled. Check if remote file exists.
--2026-02-02 11:10:14--  https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.min
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.92.184.40, 52.92.250.128, 52.218.250.168, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.92.184.40|:443... connected.
HTTP

For the graph indices, we need to extract a list of paths because 

Extract the list of paths corresponding to GRCh38

In [ ]:
%%sh 

docker run --rm \
    --volume $(pwd)/data/ref:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths \
        -x hprc-v1.1-mc-grch38.gbz \
        -L \
        -Q GRCh38 > data/ref/hprc-v1.1-mc-grch38.paths

Filter the paths list

In [ ]:
%%sh 

grep -v _decoy data/ref/hprc-v1.1-mc-grch38.paths \
    | grep -v _random \
    | grep -v chrUn_ \
    | grep -v chrEBV \
    | grep -v chrM \
    | grep -v chain_ > data/ref/hprc-v1.1-mc-grch38.paths.sub

Now we have the data downloaded and pre-processed. Check that the data directory structure matches below before moving to the next session. 

```
data
├── hprc-v1.1-mc-grch38.dist
├── hprc-v1.1-mc-grch38.fa
├── hprc-v1.1-mc-grch38.gbz
├── hprc-v1.1-mc-grch38.min
├── hprc-v1.1-mc-grch38.paths
└── hprc-v1.1-mc-grch38.paths.sub

1 directory, 6 files
```

In [22]:
%%sh 

tree data

data
├── hprc-v1.1-mc-grch38.dist
├── hprc-v1.1-mc-grch38.fa
├── hprc-v1.1-mc-grch38.gbz
├── hprc-v1.1-mc-grch38.min
├── hprc-v1.1-mc-grch38.paths
└── hprc-v1.1-mc-grch38.paths.sub

1 directory, 6 files


## Giraffe

In this section, we will run GPU accelerated Giraffe using [Parabricks Giraffe](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_giraffe.html). 

In [ ]:
%%sh 

docker run --rm --gpus all \
    --volume $(pwd)/data:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun giraffe \
        --read-group "sample_rg1" \
        --sample "sample-name" \
        --read-group-library "library" \
        --read-group-platform "platform" \
        --read-group-pu "pu" \
        --dist-name ref/hprc-v1.1-mc-grch38.dist \
        --minimizer-name ref/hprc-v1.1-mc-grch38.min \
        --gbz-name ref/hprc-v1.1-mc-grch38.gbz \
        --ref-paths ref/hprc-v1.1-mc-grch38.paths.sub \
        --in-fq sample_1.fq.gz sample_2.fq.gz \
        --out-bam out/sample.bam


[Parabricks Options Mesg]: Read group for /workdir/sample_1.fq.gz:
[Parabricks Options Mesg]: @RG\tID:sample_rg1\tLB:library\tPL:platform\tSM:sample-name\tPU:pu
[Parabricks Options Mesg]: Marking Duplicates will be run during sorting stages


[PB Info 2026-Feb-02 18:47:09] ------------------------------------------------------------------------------
[PB Info 2026-Feb-02 18:47:09] ||                 Parabricks accelerated Genomics Pipeline                 ||
[PB Info 2026-Feb-02 18:47:09] ||                              Version 4.6.0-1                             ||
[PB Info 2026-Feb-02 18:47:09] ||                              GPU-VG-Giraffe                              ||
[PB Info 2026-Feb-02 18:47:09] ------------------------------------------------------------------------------
[PB Info 2026-Feb-02 18:47:19] Loading indexes
[PB Info 2026-Feb-02 18:48:38] Giraffe CPU loading and initialization: 88.390622 seconds
[PB Info 2026-Feb-02 18:48:38] Using Parabricks batch mapper
[PB Info 2026-Feb-02 18:48:38] Running with 4 GPUs and 3 CUDA streams
[PB Info 2026-Feb-02 18:48:38] Using 64 CPU threads
[PB Info 2026-Feb-02 18:48:38] Writing to intermediate files to be sorted in later stages
[PB Info 2026-Feb-02 18:48:38] Estimating

Please visit https://docs.nvidia.com/clara/#parabricks for detailed documentation




## Pangenome-Aware DeepVariant

Next we will extract the reference path using [vg](https://github.com/vgteam/vg) paths and write the output to a FASTA file so we can use standard variant callers downstream. 

In [ ]:
%%sh 

# Extract the sequences corrresponding to the list of paths to a FASTA file
docker run --rm \
    --volume $(pwd)/data/ref:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths \
        -x hprc-v1.1-mc-grch38.gbz \
        -p hprc-v1.1-mc-grch38.paths.sub \
        -F > data/ref/hprc-v1.1-mc-grch38.fa

Index the FASTA using samtools faidx

In [26]:
%%sh 

docker run --rm \
    --volume $(pwd)/data/ref:/workdir \
    --workdir /workdir \
    biocontainers/samtools:v1.9-4-deb_cv1 \
    samtools faidx hprc-v1.1-mc-grch38.fa

GPU accelerated [Parabricks Pangenome-Aware DeepVariant](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_pangenome_aware_deepvariant.html). 

In [27]:
%%sh 

docker run --rm --gpus all \
    --volume $(pwd)/data:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun pangenome_aware_deepvariant \
    --ref ref/hprc-v1.1-mc-grch38.fa \
    --pangenome ref/hprc-v1.1-mc-grch38.gbz \
    --in-bam out/sample.bam \
    --out-variants out/sample.vcf

[Parabricks Options Mesg]: Setting --num-streams-per-gpu based on available device memory.
[Parabricks Options Mesg]: Please consider using --run-partition for best performance with more than 2 GPUs
Detected 4 CUDA Capable device(s), considering 4 device(s)
  CUDA Driver Version / Runtime Version          13.0 / 12.9
Using model for CUDA Capability Major/Minor version number:    80


[PB Info 2026-Feb-02 19:01:06] ------------------------------------------------------------------------------
[PB Info 2026-Feb-02 19:01:06] ||                 Parabricks accelerated Genomics Pipeline                 ||
[PB Info 2026-Feb-02 19:01:06] ||                              Version 4.6.0-1                             ||
[PB Info 2026-Feb-02 19:01:06] ||                        Pangenome-Aware-Deepvariant                       ||
[PB Info 2026-Feb-02 19:01:06] ------------------------------------------------------------------------------
[PB Info 2026-Feb-02 19:01:06] Starting Pangenome-Aware Deepvariant
[PB Info 2026-Feb-02 19:01:06] Running with 4 GPU devices, each with 2 group instances and 6 workers
[PB Info 2026-Feb-02 19:01:06] Starting constructor, about to open GBZ file: /workdir/ref/hprc-v1.1-mc-grch38.gbz
[PB Info 2026-Feb-02 19:01:06] Loading GBZ file into object. This may take a while for large files...
[PB Info 2026-Feb-02 19:01:53] Loading GBZ file took 208.048982 s

/usr/local/parabricks/binaries/bin/deepsomatic 4 2 --ref /workdir/ref/hprc-v1.1-mc-grch38.fa --reads /workdir/out/sample.bam -o /workdir/out/sample.vcf -n 6 --pangenome /workdir/ref/hprc-v1.1-mc-grch38.gbz --model /usr/local/parabricks/binaries/model/80+/shortread/deepvariant_pangenome_aware.eng -long_reads --channel_insert_size --keep_legacy_allele_counter_behavior --keep_only_window_spanning_haplotypes --keep_supplementary_alignments --min_mapping_quality 0 --normalize_reads --sort_by_haplotypes --parse_sam_aux_fields
Pangenome Aware Deepvariant done, total time: 3.0 min
Please visit https://docs.nvidia.com/clara/#parabricks for detailed documentation



Let's check the VCF output using bcftools

In [31]:
%%sh 

docker run --rm \
    --volume $(pwd)/data:/workdir \
    --workdir /workdir \
    biocontainers/bcftools:v1.5_cv3 \
    bcftools view --no-header out/sample.vcf | head

GRCh38#0#chr1	23359	.	C	G	10.1	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:9:2:0,2:1:9,17,0
GRCh38#0#chr1	23395	.	A	G	6.7	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:6:2:0,2:1:5,15,0
GRCh38#0#chr1	63268	.	T	C	4.8	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:5:15:0,15:1:2,17,0
GRCh38#0#chr1	63527	.	T	C	2.6	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:3:495:369,126:0.254545:0,10,1
GRCh38#0#chr1	63792	.	G	T	19.9	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:18:258:0,256:0.992248:19,22,0
GRCh38#0#chr1	69453	.	G	A	18.4	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:16:384:1,382:0.994792:18,20,0
GRCh38#0#chr1	69511	.	A	G	15.3	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:14:541:1,540:0.998152:15,21,0
GRCh38#0#chr1	69552	.	G	C	15.6	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:15:528:0,528:1:15,21,0
GRCh38#0#chr1	69569	.	T	C	13.9	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:13:506:0,506:1:13,20,0
GRCh38#0#chr1	131536	.	C	A	0.5	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:10:11:9,2:0.181818:0,12,12


write /dev/stdout: broken pipe
